In [1]:
import pandas as pd

df = pd.read_csv("Task 3 and 4_Loan_Data.csv")

print(df.shape)
print(df.columns.tolist())
print(df.head())
print(df["default"].value_counts())  

(10000, 8)
['customer_id', 'credit_lines_outstanding', 'loan_amt_outstanding', 'total_debt_outstanding', 'income', 'years_employed', 'fico_score', 'default']
   customer_id  credit_lines_outstanding  loan_amt_outstanding  \
0      8153374                         0           5221.545193   
1      7442532                         5           1958.928726   
2      2256073                         0           3363.009259   
3      4885975                         0           4766.648001   
4      4700614                         1           1345.827718   

   total_debt_outstanding       income  years_employed  fico_score  default  
0             3915.471226  78039.38546               5         605        0  
1             8228.752520  26648.43525               2         572        1  
2             2027.830850  65866.71246               4         602        0  
3             2501.730397  74356.88347               5         612        0  
4             1768.826187  23448.32631               6 

In [2]:
from sklearn.model_selection import train_test_split

feature_cols = ["credit_lines_outstanding", "loan_amt_outstanding",
                "total_debt_outstanding", "income", "years_employed", "fico_score"]

X = df[feature_cols]
y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(round(y_train.mean(), 3), round(y_test.mean(), 3))   # default rate in each split

(8000, 6) (2000, 6)
0.185 0.185


In [3]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

model.fit(X_train, y_train)

# PD for the first 5 test borrowers — each row is [P(no default), P(default)]
print(model.predict_proba(X_test)[:5])

[[1.00000000e+00 2.84846410e-13]
 [9.99827448e-01 1.72552176e-04]
 [9.99999997e-01 2.91934443e-09]
 [9.99999999e-01 5.19246916e-10]
 [9.99999999e-01 7.12449085e-10]]


In [4]:
from sklearn.metrics import roc_auc_score, accuracy_score

pd_test   = model.predict_proba(X_test)[:, 1]   # PD for every test borrower
pred_test = model.predict(X_test)               # hard 0/1 call at the 0.5 cutoff

print("AUC:", round(roc_auc_score(y_test, pd_test), 4))
print("Accuracy:", round(accuracy_score(y_test, pred_test), 4))

AUC: 1.0
Accuracy: 0.999


In [5]:
RECOVERY_RATE = 0.10

def expected_loss(loan):
    """loan: dict with the six feature keys."""
    X_one = pd.DataFrame([loan])[feature_cols]      # one-row frame, columns in the right order
    pd_default = model.predict_proba(X_one)[0, 1]   # probability this borrower defaults
    exposure = loan["loan_amt_outstanding"]         # amount on the line
    lgd = 1 - RECOVERY_RATE                          # lose 90% if they default
    return round(pd_default * lgd * exposure, 2)

In [6]:
sample_loan = {
    "credit_lines_outstanding": 6,
    "loan_amt_outstanding": 2000,
    "total_debt_outstanding": 1000,
    "income": 1000,
    "years_employed": 10,
    "fico_score": 772,
}

pd_val = model.predict_proba(pd.DataFrame([sample_loan])[feature_cols])[0, 1]
print("PD:", round(pd_val, 4))
print("Expected loss:", expected_loss(sample_loan))

PD: 0.6145
Expected loss: 1106.04


In [7]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Trees split on raw feature thresholds, so they need NO scaling
tree = DecisionTreeClassifier(max_depth=4, random_state=42)
tree.fit(X_train, y_train)

forest = RandomForestClassifier(n_estimators=200, random_state=42)
forest.fit(X_train, y_train)

log_auc    = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
tree_auc   = roc_auc_score(y_test, tree.predict_proba(X_test)[:, 1])
forest_auc = roc_auc_score(y_test, forest.predict_proba(X_test)[:, 1])

print("Logistic Regression AUC:", round(log_auc, 4))
print("Decision Tree AUC:      ", round(tree_auc, 4))
print("Random Forest AUC:      ", round(forest_auc, 4))

# Bonus: which features actually drive default?
importances = pd.Series(forest.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nFeature importances:")
print(importances.round(3))

Logistic Regression AUC: 1.0
Decision Tree AUC:       0.9996
Random Forest AUC:       0.9999

Feature importances:
credit_lines_outstanding    0.565
total_debt_outstanding      0.296
years_employed              0.052
fico_score                  0.042
income                      0.031
loan_amt_outstanding        0.015
dtype: float64
